# Top posts by score

List the posts ranked **1–1,000** and **99,000–100,000** by archived score, with the title,
score and posting date. The score is the value Arctic Shift re-fetched about 36 hours
after posting.

- Default dataset: `askreddit_dedup`, one post per normalized title (the data the current
  models use). Switch to `askreddit_clean` below to see every post.
- Ranking: score descending; equal scores are ordered by earliest post, then `id`.
- Read-only DuckDB queries over local Parquet; no Spark.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = ROOT / "data/processed/askreddit_dedup"    # one post per title (used by the models)
# DATASET = ROOT / "data/processed/askreddit_clean"  # every post, before title deduplication
files = [str(p) for p in sorted(DATASET.rglob("*.parquet"))]
con = duckdb.connect()

top = con.execute("""
    SELECT row_number() OVER (ORDER BY score DESC, created_utc, id) AS rank, id, title, score, created_utc
    FROM read_parquet(?)
    QUALIFY rank <= 100000
    ORDER BY rank
""", [files]).df()
posted = pd.to_datetime(top["created_utc"], unit="s", utc=True)
top["posted_utc"] = posted.dt.strftime("%Y-%m-%d %H:%M")
top["posted_vietnam"] = posted.dt.tz_convert("Asia/Ho_Chi_Minh").dt.strftime("%Y-%m-%d %H:%M")
COLUMNS = ["rank", "title", "score", "posted_utc", "posted_vietnam"]
total = con.execute("SELECT count(*) FROM read_parquet(?)", [files]).fetchone()[0]
print(f"{DATASET.name}: {total:,} posts; loaded ranks 1–{len(top):,}")

## Ranks 1–1,000

In [ ]:
with pd.option_context("display.max_rows", 1000, "display.max_colwidth", 140):
    display(top.loc[top["rank"].between(1, 1000), COLUMNS].set_index("rank"))

## Ranks 99,000–100,000

In [ ]:
with pd.option_context("display.max_rows", 1001, "display.max_colwidth", 140):
    display(top.loc[top["rank"].between(99000, 100000), COLUMNS].set_index("rank"))

## Where the top 100,000 ends

Scores are heavily tied at the bottom of the top 100,000, so the cut-off falls inside a tie.

In [ ]:
at_rank = top.set_index("rank")["score"]
cut = int(at_rank[100000])
above, tied = con.execute(
    "SELECT count(*) FILTER (WHERE score > ?), count(*) FILTER (WHERE score = ?) FROM read_parquet(?)", [cut, cut, files]
).fetchone()
print("Score at rank 1 / 1,000 / 99,000 / 100,000:", *(f"{int(at_rank[r]):,}" for r in (1, 1000, 99000, 100000)))
print(f"{tied:,} posts have score {cut}; they occupy ranks {above + 1:,}–{above + tied:,} of {total:,}.")
print("Within that tie the order (earliest post first) is arbitrary, so 'top 100,000' is really 'score >= "
      f"{cut}' ({above + tied:,} posts, {100 * (above + tied) / total:.1f}% of all posts) cut at an arbitrary point.")